# Baseline Relation Extraction from MediaWiki XML

This notebook implements the first baseline model for the character relationship Knowledge Graph project:

**TF-IDF features + Logistic Regression**

Pipeline

1. Set the XML input path and editable relationship-label list.
2. Parse the MediaWiki XML and extract each character page using the page `<title>` as the character name and the revision `<text>` as the character text.
3. Clean wiki markup.
4. Build a character gazetteer from page titles.
5. Generate candidate character pairs.
6. Create weak labels from infobox fields and relationship keywords.
7. Train and evaluate TF-IDF + Logistic Regression baseline variations.
8. Predict relationships for all candidates.
9. Aggregate predictions into Knowledge Graph edges.
10. Save all outputs to a folder called `baseline/`.
11. Create an interactive PyVis graph

The generated labels are weak labels. Since these labels are generated automatically from rules and infobox fields, the evaluation measures how well the model learns the weak labeling scheme rather than fully verified human-annotated relationships.

## 1. Configuration

In [1]:
from pathlib import Path

# Input XML.
XML_PATH = Path("data/bookworm_09062026.xml")

# Every generated file will be written under this directory.
BASELINE_DIR = Path("baseline")
BASELINE_DIR.mkdir(parents=True, exist_ok=True)

# Relationship labels for this baseline. "no_relation" is needed as the negative class.
RELATIONSHIPS = [
    "family",
    "romantic",
    "friend_ally",
    "service_retainer",
    "enemy_rival",
    "no_relation",
]

NO_RELATION_LABEL = "no_relation"
POSITIVE_RELATIONS = [label for label in RELATIONSHIPS if label != NO_RELATION_LABEL]

RANDOM_SEED = 42
TEST_SIZE = 0.15
DEV_SIZE = 0.15
MAX_NEGATIVE_RATIO = 2.0
MIN_CONTEXT_CHARS = 25
EDGE_CONFIDENCE_THRESHOLD = 0.95 # predictions_all_best_baseline.csv

print(f"XML path: {XML_PATH.resolve()}")
print(f"Output folder: {BASELINE_DIR.resolve()}")
print(f"Relationship labels: {RELATIONSHIPS}")

XML path: E:\Natural Language Processing\Project 2\data\bookworm_09062026.xml
Output folder: E:\Natural Language Processing\Project 2\baseline
Relationship labels: ['family', 'romantic', 'friend_ally', 'service_retainer', 'enemy_rival', 'no_relation']


## 2. Imports

In [2]:
import html
import json
import math
import re
import warnings
import xml.etree.ElementTree as ET
from collections import Counter, defaultdict
from pathlib import Path

import joblib
import numpy as np
import pandas as pd

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_recall_fscore_support,
)
from sklearn.model_selection import GroupShuffleSplit, train_test_split
from sklearn.multiclass import OneVsRestClassifier
from sklearn.pipeline import Pipeline

warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.max_colwidth", 140)
np.random.seed(RANDOM_SEED)

## 3. Parse the MediaWiki XML

MediaWiki exports store the page title and page text as XML elements. This cell extracts them as:

- `character_name` = page `<title>`
- `character_text` = revision `<text>`

Characters pages that have less than 2000 characters are removed

In [3]:
def get_xml_namespace(tag: str) -> str:
    """Return namespace prefix in ElementTree format, e.g. '{...}', or empty string."""
    if tag.startswith("{"):
        return tag.split("}", 1)[0] + "}"
    return ""


def parse_mediawiki_xml(xml_path: Path) -> pd.DataFrame:
    """Parse a MediaWiki XML export into one row per page."""
    xml_path = Path(xml_path)
    if not xml_path.exists():
        raise FileNotFoundError(f"XML file not found: {xml_path}")

    pages = []
    context = ET.iterparse(str(xml_path), events=("start", "end"))
    _, root = next(context)
    ns = get_xml_namespace(root.tag)

    for event, elem in context:
        if event == "end" and elem.tag == f"{ns}page":
            title = (elem.findtext(f"{ns}title") or "").strip()
            ns_id = (elem.findtext(f"{ns}ns") or "").strip()
            page_id = (elem.findtext(f"{ns}id") or "").strip()
            revision = elem.find(f"{ns}revision")
            text = ""
            timestamp = ""
            if revision is not None:
                text = revision.findtext(f"{ns}text") or ""
                timestamp = revision.findtext(f"{ns}timestamp") or ""

            is_redirect = elem.find(f"{ns}redirect") is not None or text.lstrip().upper().startswith("#REDIRECT")

            pages.append(
                {
                    "page_id": page_id,
                    "namespace": ns_id,
                    "character_name": title,
                    "character_text": text,
                    "revision_timestamp": timestamp,
                    "is_redirect": is_redirect,
                    "text_length": len(text),
                }
            )
            elem.clear()
            root.clear()

    return pd.DataFrame(pages)


pages_df = parse_mediawiki_xml(XML_PATH)

pages_df = pages_df[
    (pages_df["namespace"] == "0")
    & (~pages_df["is_redirect"])
    & (pages_df["text_length"] >= 2000)
].copy()

pages_df = pages_df.sort_values("character_name").reset_index(drop=True)
pages_df.to_csv(BASELINE_DIR / "pages.csv", index=False)

print(f"Parsed character pages with text_length >= 2000: {len(pages_df):,}")
pages_df[["page_id", "character_name", "text_length", "revision_timestamp"]].head(10)

Parsed character pages with text_length >= 2000: 257


,page_id,character_name,text_length,revision_timestamp
0,3753,Achim,3588,2025-12-18T23:44:30Z
1,3880,Adelbert,9599,2026-03-24T02:58:11Z
2,3934,Adolphine,19043,2026-05-03T19:26:14Z
3,5956,Adrett,3808,2025-05-23T14:26:44Z
4,7262,Aeussewahl,2196,2026-03-08T01:09:51Z
5,7266,Albsenti,2682,2026-03-08T02:57:20Z
6,3836,Alexis,7005,2026-02-26T22:10:43Z
7,3615,Anastasius,6855,2026-03-12T03:17:38Z
8,6591,Andrea,2153,2026-04-13T23:37:05Z
9,1287,Angelica,8337,2026-04-18T16:50:45Z


## 4. Wikitext cleaning and character gazetteer

The character gazetteer is built from page titles. The cleaning functions remove common wiki markup while preserving readable text for sentence-level candidate generation.

In [4]:
def normalize_title(title: str) -> str:
    """Normalize a MediaWiki title for matching."""
    title = html.unescape(str(title or ""))
    title = title.replace("_", " ").strip()
    title = re.sub(r"\s+", " ", title)
    return title


def strip_anchor(title: str) -> str:
    """Remove a #section anchor from a wiki title."""
    return normalize_title(str(title).split("#", 1)[0])


def extract_wikilinks(wikitext: str):
    """Extract wiki links as (target, display_text) pairs."""
    if not isinstance(wikitext, str):
        return []
    links = []
    for match in re.finditer(r"\[\[([^\]|#]+)(?:#[^\]|]*)?(?:\|([^\]]+))?\]\]", wikitext):
        target = strip_anchor(match.group(1))
        display_text = normalize_title(match.group(2) if match.group(2) else target)
        if target:
            links.append((target, display_text))
    return links


def remove_templates(text: str) -> str:
    """Remove balanced-looking {{...}} templates iteratively.

    This is a lightweight regex cleaner for a baseline notebook. It is not a full
    MediaWiki parser, but it is enough for candidate extraction and TF-IDF text.
    """
    pattern = re.compile(r"\{\{[^{}]*\}\}", flags=re.DOTALL)
    previous = None
    while previous != text:
        previous = text
        text = pattern.sub(" ", text)
    return text


def clean_wikitext(wikitext: str) -> str:
    """Convert raw wikitext into plain-ish text suitable for TF-IDF."""
    if not isinstance(wikitext, str):
        return ""

    text = html.unescape(wikitext)
    text = re.sub(r"<!--.*?-->", " ", text, flags=re.DOTALL)
    text = re.sub(r"<ref\b[^>/]*/>", " ", text, flags=re.IGNORECASE)
    text = re.sub(r"<ref\b[^>]*>.*?</ref>", " ", text, flags=re.IGNORECASE | re.DOTALL)
    text = re.sub(r"<gallery\b[^>]*>.*?</gallery>", " ", text, flags=re.IGNORECASE | re.DOTALL)
    text = re.sub(r"\[\[(?:Category|File|Image):[^\]]+\]\]", " ", text, flags=re.IGNORECASE)

    # Convert headings like {{h1|Story}} or {{h2|[[Part 4 Volume 3]]}} to text before template removal.
    text = re.sub(r"\{\{h[12]\|([^}|]+).*?\}\}", lambda m: f"\n{clean_wikitext(m.group(1))}\n", text, flags=re.IGNORECASE | re.DOTALL)

    # Convert wiki links to display text.
    text = re.sub(
        r"\[\[([^\]|#]+)(?:#[^\]|]*)?(?:\|([^\]]+))?\]\]",
        lambda m: normalize_title(m.group(2) if m.group(2) else m.group(1)),
        text,
    )

    text = re.sub(r"\[https?://[^\s\]]+\s+([^\]]+)\]", r"\1", text)
    text = re.sub(r"\[https?://[^\]]+\]", " ", text)
    text = re.sub(r"<br\s*/?>", "\n", text, flags=re.IGNORECASE)
    text = re.sub(r"<[^>]+>", " ", text)
    text = remove_templates(text)
    text = text.replace(chr(39) * 3, "").replace(chr(39) * 2, "")
    text = re.sub(r"={2,}[^=]+={2,}", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text


characters = sorted({normalize_title(name) for name in pages_df["character_name"].dropna() if normalize_title(name)})
character_set = set(characters)

# Fast matcher for page-title character names. This avoids scanning every
# character name with a separate regex for every sentence.
def build_character_pattern(names: list[str]) -> re.Pattern:
    names = [name for name in names if len(name) >= 3]
    names = sorted(names, key=len, reverse=True)
    if not names:
        return re.compile(r"a^")  # matches nothing
    return re.compile(r"(?<![A-Za-z])(" + "|".join(re.escape(name) for name in names) + r")(?![A-Za-z])", flags=re.IGNORECASE)

CHARACTER_PATTERN = build_character_pattern(characters)
CHARACTER_LOOKUP = {name.casefold(): name for name in characters}

# Store a simple gazetteer for later inspection.
gazetteer_df = pd.DataFrame({"character_name": characters})
gazetteer_df.to_csv(BASELINE_DIR / "character_gazetteer.csv", index=False)

print(f"Characters in gazetteer: {len(gazetteer_df):,}")
display(gazetteer_df.head(10))

Characters in gazetteer: 257


,character_name
0,Achim
1,Adelbert
2,Adolphine
3,Adrett
4,Aeussewahl
5,Albsenti
6,Alexis
7,Anastasius
8,Andrea
9,Angelica


## 5. Extract weak relationship signals from infoboxes

The MediaWiki character template often contains family fields such as `family/Father`, `family/Sister`, or `family/Spouse`. These provide useful weak labels.

This baseline maps:

- `family/Spouse`, `wife`, `husband`, `betrothed`, etc. → `romantic`
- other `family/...` fields → `family`

Only linked character names that also appear in the exported page-title gazetteer are kept.

In [5]:
def extract_balanced_template(wikitext: str, template_name: str = "Character") -> str:
    """Return the first balanced {{Character ...}} template block, if found."""
    if not isinstance(wikitext, str):
        return ""
    match = re.search(r"\{\{\s*" + re.escape(template_name) + r"\b", wikitext, flags=re.IGNORECASE)
    if not match:
        return ""

    start = match.start()
    i = start
    depth = 0
    while i < len(wikitext) - 1:
        two = wikitext[i : i + 2]
        if two == "{{":
            depth += 1
            i += 2
            continue
        if two == "}}":
            depth -= 1
            i += 2
            if depth == 0:
                return wikitext[start:i]
            continue
        i += 1
    return ""


def parse_template_fields(template_text: str) -> dict:
    """Parse simple |key=value lines from a template.
    """
    fields = {}
    current_key = None
    for raw_line in template_text.splitlines():
        line = raw_line.strip()
        if line.startswith("|") and "=" in line:
            key, value = line[1:].split("=", 1)
            current_key = key.strip()
            fields[current_key] = value.strip()
        elif current_key and line and not line.startswith("}}"):  # continuation line
            fields[current_key] += " " + line
    return fields


def relation_from_infobox_field(key: str, value: str) -> str | None:
    """Map an infobox field key/value pair to one of the relationship labels."""
    key_l = key.lower().strip()
    value_l = value.lower().strip()

    romantic_markers = [
        "spouse",
        "wife",
        "husband",
        "lover",
        "fiance",
        "fiancée",
        "betrothed",
    ]

    service_retainer_markers = [
        "retainer",
        "retainers",
        "attendant",
        "attendants",
        "guard knight",
        "guard knights",
        "scholar",
        "scholars",
        "serves",
        "served",
        "serving",
    ]

    if any(marker in key_l for marker in romantic_markers):
        return "romantic" if "romantic" in RELATIONSHIPS else None

    if key_l.startswith("family/") or key_l in {"familytree", "relatives"}:
        return "family" if "family" in RELATIONSHIPS else None

    if any(marker in key_l for marker in service_retainer_markers):
        return "service_retainer" if "service_retainer" in RELATIONSHIPS else None

    if key_l == "occupation" and any(marker in value_l for marker in service_retainer_markers):
        return "service_retainer" if "service_retainer" in RELATIONSHIPS else None

    return None


def extract_infobox_relation_edges(row: pd.Series) -> list[dict]:
    """Extract weakly labeled relation examples from one page's Character infobox."""
    head = row["character_name"]
    template = extract_balanced_template(row["character_text"], "Character")
    fields = parse_template_fields(template)
    examples = []

    for key, value in fields.items():
        label = relation_from_infobox_field(key, value)
        if label is None:
            continue

        for target, display_text in extract_wikilinks(value):
            if target in character_set and target != head:
                evidence = clean_wikitext(value)
                examples.append(
                    {
                        "head": head,
                        "tail": target,
                        "context": f"Infobox field {key}: {evidence}",
                        "section": "infobox",
                        "source_type": "infobox",
                        "weak_label": label,
                        "weak_label_source": f"infobox:{key}",
                    }
                )
    return examples


infobox_examples = []
for _, row in pages_df.iterrows():
    infobox_examples.extend(extract_infobox_relation_edges(row))

infobox_df = pd.DataFrame(infobox_examples)
print(f"Infobox weak relation examples: {len(infobox_df):,}")
if len(infobox_df):
    display(infobox_df.head(10))
    display(infobox_df["weak_label"].value_counts())

Infobox weak relation examples: 1,177


,head,tail,context,section,source_type,weak_label,weak_label_source
0,Adelbert,Bonifatius,Infobox field family/Brother: Bonifatius Bezewanst (In-Law),infobox,infobox,family,infobox:family/Brother
1,Adelbert,Bezewanst,Infobox field family/Brother: Bonifatius Bezewanst (In-Law),infobox,infobox,family,infobox:family/Brother
2,Adelbert,Irmhilde,"Infobox field family/Sister: Irmhilde (Half-Sister, Deceased) Unnamed Woman (Half-Sister, Deceased)",infobox,infobox,family,infobox:family/Sister
3,Adelbert,Veronica,"Infobox field family/Spouse: Veronica (First Wife) Irmhilde (Engaged, Deceased)",infobox,infobox,romantic,infobox:family/Spouse
4,Adelbert,Irmhilde,"Infobox field family/Spouse: Veronica (First Wife) Irmhilde (Engaged, Deceased)",infobox,infobox,romantic,infobox:family/Spouse
5,Adelbert,Constanze,Infobox field family/Daughter: Constanze Georgine Florencia (In-Law),infobox,infobox,family,infobox:family/Daughter
6,Adelbert,Georgine,Infobox field family/Daughter: Constanze Georgine Florencia (In-Law),infobox,infobox,family,infobox:family/Daughter
7,Adelbert,Florencia,Infobox field family/Daughter: Constanze Georgine Florencia (In-Law),infobox,infobox,family,infobox:family/Daughter
8,Adelbert,Sylvester,Infobox field family/Son: Sylvester Ferdinand Aub Frenbeltag (In-Law) Gieselfried (In-Law),infobox,infobox,family,infobox:family/Son
9,Adelbert,Ferdinand,Infobox field family/Son: Sylvester Ferdinand Aub Frenbeltag (In-Law) Gieselfried (In-Law),infobox,infobox,family,infobox:family/Son


weak_label
family              998
service_retainer     93
romantic             86
Name: count, dtype: int64

## 6. Generate sentence-level candidate pairs

For each character page, this cell:

1. cleans the page text,
2. splits it into sentence-like contexts,
3. finds mentions of other exported character-page titles,
4. creates a candidate pair `(page character, mentioned character)`, and
5. assigns a weak label using relationship keywords.

These labels are noisy and should be treated as baseline/distant-supervision labels.

In [6]:
RELATION_PATTERNS = {
    "family": [
        r"\bfather\b", r"\bmother\b", r"\bparent\b", r"\bparents\b", r"\bson\b", r"\bdaughter\b",
        r"\bsibling\b", r"\bbrother\b", r"\bsister\b", r"\buncle\b", r"\baunt\b", r"\bcousin\b",
        r"\bgrandfather\b", r"\bgrandmother\b", r"\brelative\b", r"\bin-law\b", r"\bniece\b", r"\bnephew\b",
    ],
    "romantic": [
        r"\bwife\b", r"\bhusband\b", r"\bspouse\b", r"\bmarried\b", r"\bmarriage\b",
        r"\bengaged\b", r"\bengagement\b", r"\bbetrothed\b", r"\bfianc[eé]\b",
        r"\blover\b", r"\blove interest\b", r"\bin love\b", r"\bromantic\b",
    ],
    "friend_ally": [
        r"\bfriend\b", r"\bfriends\b", r"\bfriendship\b",
        r"\bally\b", r"\ballies\b", r"\ballied\b", r"\bcompanion\b",
        r"\bclose friend\b", r"\bbest friend\b", r"\btrusted friend\b", r"\bconfidant\b",
    ],
    "service_retainer": [
        r"\bretainer\b", r"\bretainers\b", r"\bguard knight\b", r"\bguard knights\b",
        r"\battendant\b", r"\battendants\b",
        r"\bscholar\b", r"\bscholars\b",
        r"\bserved\b", r"\bserves\b", r"\bserving\b",
        r"\bin .* service\b", r"\bhead attendant\b", r"\bhead scholar\b", r"\bhead guard knight\b",
    ],
    "enemy_rival": [
        r"\benemy\b", r"\benemies\b", r"\brival\b", r"\brivals\b", r"\bopponent\b", r"\bopposes\b",
        r"\bbetray\b", r"\bbetrayed\b", r"\bkill(?:ed|s)?\b", r"\bexecute(?:d|s)?\b", r"\bhate(?:d|s)?\b",
        r"\babuse(?:d|s)?\b", r"\bjealous\b", r"\bpersecution\b", r"\battack(?:ed|s)?\b",
    ],
}

# Keep only patterns for labels present in RELATIONSHIPS.
RELATION_PATTERNS = {label: pats for label, pats in RELATION_PATTERNS.items() if label in RELATIONSHIPS}
RELATION_PRIORITY = ["romantic", "family", "enemy_rival", "service_retainer", "friend_ally"]
RELATION_PRIORITY = [label for label in RELATION_PRIORITY if label in RELATIONSHIPS]


def split_wikitext_into_sections(wikitext: str) -> list[tuple[str, str]]:
    """Split raw wikitext by Fandom-style {{h1|...}} / {{h2|...}} headings."""
    if not isinstance(wikitext, str):
        return [("lead", "")]

    heading_re = re.compile(r"\{\{h[12]\|([^}|]+)(?:\|[^}]*)?\}\}", flags=re.IGNORECASE | re.DOTALL)
    sections = []
    current_section = "lead"
    last = 0

    for match in heading_re.finditer(wikitext):
        if match.start() > last:
            sections.append((current_section, wikitext[last: match.start()]))
        current_section = clean_wikitext(match.group(1)) or "section"
        last = match.end()

    sections.append((current_section, wikitext[last:]))
    return sections


def split_sentences(text: str) -> list[str]:
    """Simple sentence splitter for cleaned wiki text."""
    text = re.sub(r"\s+", " ", text).strip()
    if not text:
        return []
    pieces = re.split(r"(?<=[.!?])\s+(?=[A-Z0-9'\"“])", text)
    return [piece.strip() for piece in pieces if len(piece.strip()) >= MIN_CONTEXT_CHARS]


def name_in_text(name: str, text: str) -> bool:
    """Case-insensitive character-name match with letter boundaries."""
    pattern = r"(?<![A-Za-z])" + re.escape(name) + r"(?![A-Za-z])"
    return re.search(pattern, text, flags=re.IGNORECASE) is not None


def find_mentioned_characters(text: str, head: str) -> list[str]:
    """Find other character names from the gazetteer mentioned in text."""
    found = []
    for match in CHARACTER_PATTERN.finditer(text):
        canonical = CHARACTER_LOOKUP.get(match.group(0).casefold())
        if canonical and canonical != head:
            found.append(canonical)
    return sorted(set(found))


def infer_weak_label_from_text(text: str, section: str = "") -> tuple[str, str]:
    """Infer a weak label from keyword patterns."""
    full_text = f"{section} {text}".lower()
    scores = {}
    matches = {}

    for label, patterns in RELATION_PATTERNS.items():
        label_matches = [pat for pat in patterns if re.search(pat, full_text, flags=re.IGNORECASE)]
        if label_matches:
            scores[label] = len(label_matches)
            matches[label] = label_matches

    if not scores:
        return NO_RELATION_LABEL, "keyword:none"

    # Highest score wins. Priority order breaks ties.
    max_score = max(scores.values())
    best_labels = [label for label, score in scores.items() if score == max_score]
    for label in RELATION_PRIORITY:
        if label in best_labels:
            return label, f"keyword:{','.join(matches[label][:3])}"
    return best_labels[0], f"keyword:{','.join(matches[best_labels[0]][:3])}"


def mark_entities(context: str, head: str, tail: str) -> str:
    """Add explicit entity markers for the target pair."""
    marked = context

    tail_pattern = re.compile(r"(?<![A-Za-z])" + re.escape(tail) + r"(?![A-Za-z])", flags=re.IGNORECASE)
    head_pattern = re.compile(r"(?<![A-Za-z])" + re.escape(head) + r"(?![A-Za-z])", flags=re.IGNORECASE)

    marked = tail_pattern.sub(lambda m: f"[TAIL] {m.group(0)} [/TAIL]", marked, count=1)

    if head_pattern.search(marked):
        marked = head_pattern.sub(lambda m: f"[HEAD] {m.group(0)} [/HEAD]", marked, count=1)
    else:
        marked = f"[HEAD] {head} [/HEAD] {marked}"

    return marked


def make_model_text(row: pd.Series, use_markers: bool = True, include_metadata: bool = True) -> str:
    """Build the text input for TF-IDF."""
    context = row["context"]
    if use_markers:
        context = mark_entities(context, row["head"], row["tail"])
    if include_metadata:
        return f"section={row['section']} source={row['source_type']} {context}"
    return context


def generate_sentence_candidates(pages: pd.DataFrame) -> pd.DataFrame:
    examples = []

    for _, row in pages.iterrows():
        head = row["character_name"]
        for section, raw_section in split_wikitext_into_sections(row["character_text"]):
            cleaned_section = clean_wikitext(raw_section)
            for sentence in split_sentences(cleaned_section):
                mentioned = find_mentioned_characters(sentence, head)
                for tail in mentioned:
                    weak_label, weak_source = infer_weak_label_from_text(sentence, section)
                    examples.append(
                        {
                            "head": head,
                            "tail": tail,
                            "context": sentence,
                            "section": section,
                            "source_type": "prose",
                            "weak_label": weak_label,
                            "weak_label_source": weak_source,
                        }
                    )

    return pd.DataFrame(examples)


sentence_df = generate_sentence_candidates(pages_df)
print(f"Sentence-level candidate examples: {len(sentence_df):,}")
if len(sentence_df):
    display(sentence_df.head(10))
    display(sentence_df["weak_label"].value_counts())

Sentence-level candidate examples: 4,858


,head,tail,context,section,source_type,weak_label,weak_label_source
0,Achim,Egon,"Alongside Egon, a fellow priest, Achim is assigned to spend a winter in Hasse teaching the local commoners how to interact with nobles.",lead,prose,no_relation,keyword:none
1,Achim,Rozemyne,"Later, he assists Rozemyne and her Gutenbergs in establishing new paper-making workshops in the provinces of Ehrenfest.",lead,prose,no_relation,keyword:none
2,Achim,Rozemyne,"After the former mayor of Hasse is executed for treason against the archduke, Rozemyne is determined to educate the local commoners to p...",Part 3 Volume 3,prose,enemy_rival,keyword:\bexecute(?:d|s)?\b
3,Achim,Rozemyne,"She is especially worried about Richt, the new mayor, who seems to have little knowledge of noble euphemisms. (For example, he offers Ro...",Part 3 Volume 4,prose,no_relation,keyword:none
4,Achim,Egon,"In the autumn following Hasse's first year of punishment, she assigns the gray priests Achim and Egon to stay in the village over the wi...",Part 3 Volume 5,prose,no_relation,keyword:none
5,Achim,Rozemyne,"While in Hasse, the priests are also tasked with gathering local folktales for ""Operation Grimm,"" which Rozemyne plans to one day compil...",Part 3 Volume 5,prose,no_relation,keyword:none
6,Achim,Egon,Although Achim and Egon initially struggle to adjust to the unclean living conditions of Hasse's winter mansion and the rough table mann...,Part 3 Volume 5,prose,no_relation,keyword:none
7,Achim,Rozemyne,"When Rozemyne returns from her first term at the Royal Academy, she once again assigns Achim to travel to the provinces, this time as pa...",Part 4 Volume 3,prose,no_relation,keyword:none
8,Adelbert,Sylvester,"After his passing, leadership of the duchy passed to his eldest son and heir, Sylvester.",lead,prose,family,keyword:\bson\b
9,Adelbert,Constanze,"With his first wife Veronica, Adelbert had three children: Georgine, Constanze and Sylvester.",lead,prose,romantic,keyword:\bwife\b


weak_label
no_relation         2577
family               900
romantic             616
service_retainer     531
enemy_rival          163
friend_ally           71
Name: count, dtype: int64

## 7. Build the baseline training dataset

This cell combines infobox examples and prose examples, deduplicates them, balances the number of negative examples, and creates two text versions:

- `text_basic`: context only
- `text_marked`: context with `[HEAD]` / `[TAIL]` markers plus section/source metadata

The logistic regression baseline will be trained on both versions as two baseline variations.

In [7]:
candidate_parts = []
if len(infobox_df):
    candidate_parts.append(infobox_df)
if len(sentence_df):
    candidate_parts.append(sentence_df)

if not candidate_parts:
    raise ValueError("No candidate examples were generated. Check the XML path and character-page filter.")

candidates_df = pd.concat(candidate_parts, ignore_index=True)
candidates_df = candidates_df.drop_duplicates(subset=["head", "tail", "context", "source_type"]).reset_index(drop=True)

# Keep only labels included in the configured relationship list.
candidates_df = candidates_df[candidates_df["weak_label"].isin(RELATIONSHIPS)].copy()

# Balance negatives against positives so the model does not learn to predict only no_relation.
pos_df = candidates_df[candidates_df["weak_label"] != NO_RELATION_LABEL].copy()
neg_df = candidates_df[candidates_df["weak_label"] == NO_RELATION_LABEL].copy()

if len(pos_df) == 0:
    raise ValueError("No positive relationship examples were generated. Add more relation keywords or use manual labels.")

max_negatives = int(MAX_NEGATIVE_RATIO * len(pos_df))
if len(neg_df) > max_negatives:
    neg_df = neg_df.sample(n=max_negatives, random_state=RANDOM_SEED)

candidates_df = pd.concat([pos_df, neg_df], ignore_index=True)
candidates_df = candidates_df.sample(frac=1.0, random_state=RANDOM_SEED).reset_index(drop=True)

candidates_df["label"] = candidates_df["weak_label"]
candidates_df["pair_id"] = candidates_df["head"] + " || " + candidates_df["tail"]
candidates_df["candidate_id"] = [f"cand_{i:06d}" for i in range(len(candidates_df))]
candidates_df["text_basic"] = candidates_df["context"]
candidates_df["text_marked"] = candidates_df.apply(lambda row: make_model_text(row, use_markers=True, include_metadata=True), axis=1)

candidates_df.to_csv(BASELINE_DIR / "candidate_examples.csv", index=False)

label_counts = candidates_df["label"].value_counts().rename_axis("label").reset_index(name="count")
label_counts.to_csv(BASELINE_DIR / "label_distribution.csv", index=False)

print(f"Training candidates after balancing: {len(candidates_df):,}")
display(label_counts)
display(candidates_df[["candidate_id", "head", "tail", "label", "source_type", "context"]].head(10))

Training candidates after balancing: 6,031


,label,count
0,no_relation,2577
1,family,1897
2,romantic,702
3,service_retainer,621
4,enemy_rival,163
5,friend_ally,71


,candidate_id,head,tail,label,source_type,context
0,cand_000000,Georgine,Sylvester,family,prose,"She is the mother of Detlinde, Sylvester's older sister and a former archduke candidate of Ehrenfest."
1,cand_000001,Magdalena,Werdekraf,romantic,prose,"She also often trained with her brother Werdekraf before she moved to live at the Royal Palace, following her marriage."
2,cand_000002,Arno,Ferdinand,enemy_rival,prose,"Ferdinand, displeased at Arno's actions, had Arno killed."
3,cand_000003,Magdalena,Hildebrand,no_relation,prose,The boy is Hildebrand and was raised to become a vassal to whichever of his two older brothers would win the position of successor.
4,cand_000004,Wilfried,Veronica,family,prose,Upon Veronica's imprisonment Wilfried was told that his grandmother had fallen ill and had been moved to a far away place to recover.
5,cand_000005,Lasfam,Ferdinand,service_retainer,prose,"When Ferdinand personally asked him to do as the others did so Lasfam wouldn't have to suffer so much, Lasfam refused, stating that doin..."
6,cand_000006,Eckhart,Lamprecht,family,prose,"Despite despising Veroncia as well, his father still didn't want this to come to pass and when he learned of Eckhart's plans intervened ..."
7,cand_000007,Lungtase,Raufereg,family,prose,"Her less well behaved brother Raufereg has to stay home, since his parents don't want to risk embarrassing themselves and by extension t..."
8,cand_000008,Gloria,Rozemyne,no_relation,prose,"When Rozemyne enters noble society, Gloria holds her in contempt and often slanders her reputation at tea parties and social gatherings."
9,cand_000009,Georgine,Veronica,service_retainer,prose,"While Veronica was frozen in terror, one of her attendants had immediately sprung into action and administered an antidote."


## 8. Optional manual-label override

For a final report, manually check labels. One simple workflow:

1. Open `baseline/candidate_examples.csv`.
2. Correct the `label` column.
3. Save the corrected file as `baseline/manual_labels.csv`.
4. Re-run the notebook from this cell.

If `baseline/manual_labels.csv` exists, this cell uses it instead of the weak labels.

In [8]:
MANUAL_LABELS_PATH = BASELINE_DIR / "manual_labels.csv"

if MANUAL_LABELS_PATH.exists():
    manual_df = pd.read_csv(MANUAL_LABELS_PATH)
    required_cols = {"candidate_id", "label"}
    if not required_cols.issubset(manual_df.columns):
        raise ValueError(f"Manual labels file must contain columns: {required_cols}")

    label_map = dict(zip(manual_df["candidate_id"], manual_df["label"]))
    candidates_df["label"] = candidates_df["candidate_id"].map(label_map).fillna(candidates_df["label"])
    candidates_df = candidates_df[candidates_df["label"].isin(RELATIONSHIPS)].copy()
    print(f"Manual labels loaded from {MANUAL_LABELS_PATH}")
else:
    print("No manual label file found. Using weak labels generated by the notebook.")

display(candidates_df["label"].value_counts().rename_axis("label").reset_index(name="count"))

No manual label file found. Using weak labels generated by the notebook.


,label,count
0,no_relation,2577
1,family,1897
2,romantic,702
3,service_retainer,621
4,enemy_rival,163
5,friend_ally,71


## 9. Train/dev/test split by character pair

The split uses `pair_id = head || tail` as the group. This reduces leakage, because the same character pair should not appear in both train and test.

In [9]:
def group_split_dataframe(df: pd.DataFrame, group_col: str = "pair_id"):
    """Split into train/dev/test using grouped splits.

    If the grouped split fails due to very small data, fall back to stratified random splits.
    """
    df = df.copy().reset_index(drop=True)
    groups = df[group_col]

    try:
        splitter1 = GroupShuffleSplit(n_splits=1, test_size=TEST_SIZE, random_state=RANDOM_SEED)
        train_dev_idx, test_idx = next(splitter1.split(df, groups=groups))
        train_dev_df = df.iloc[train_dev_idx].copy()
        test_df = df.iloc[test_idx].copy()

        relative_dev_size = DEV_SIZE / (1.0 - TEST_SIZE)
        splitter2 = GroupShuffleSplit(n_splits=1, test_size=relative_dev_size, random_state=RANDOM_SEED)
        train_idx, dev_idx = next(splitter2.split(train_dev_df, groups=train_dev_df[group_col]))
        train_df = train_dev_df.iloc[train_idx].copy()
        dev_df = train_dev_df.iloc[dev_idx].copy()
        split_method = "grouped_by_pair"
    except Exception as exc:
        print(f"Grouped split failed ({exc}). Falling back to stratified random split.")
        train_dev_df, test_df = train_test_split(
            df,
            test_size=TEST_SIZE,
            random_state=RANDOM_SEED,
            stratify=df["label"] if df["label"].nunique() > 1 else None,
        )
        relative_dev_size = DEV_SIZE / (1.0 - TEST_SIZE)
        train_df, dev_df = train_test_split(
            train_dev_df,
            test_size=relative_dev_size,
            random_state=RANDOM_SEED,
            stratify=train_dev_df["label"] if train_dev_df["label"].nunique() > 1 else None,
        )
        split_method = "stratified_random"

    return (
        train_df.reset_index(drop=True),
        dev_df.reset_index(drop=True),
        test_df.reset_index(drop=True),
        split_method,
    )


train_df, dev_df, test_df, split_method = group_split_dataframe(candidates_df)

if train_df["label"].nunique() < 2:
    raise ValueError("Training split has fewer than two classes. Add more labels/examples or reduce filtering.")

train_df.to_csv(BASELINE_DIR / "train.csv", index=False)
dev_df.to_csv(BASELINE_DIR / "dev.csv", index=False)
test_df.to_csv(BASELINE_DIR / "test.csv", index=False)

print(f"Split method: {split_method}")
print(f"Train: {len(train_df):,} | Dev: {len(dev_df):,} | Test: {len(test_df):,}")

split_summary = pd.concat(
    [
        train_df["label"].value_counts().rename("train"),
        dev_df["label"].value_counts().rename("dev"),
        test_df["label"].value_counts().rename("test"),
    ],
    axis=1,
).fillna(0).astype(int)
split_summary.to_csv(BASELINE_DIR / "split_label_distribution.csv")
display(split_summary)

Split method: grouped_by_pair
Train: 4,251 | Dev: 915 | Test: 865


,train,dev,test
label,,,
no_relation,1772,405,400
family,1366,274,257
romantic,513,108,81
service_retainer,442,95,84
enemy_rival,114,22,27
friend_ally,44,11,16


## 10. Train TF-IDF + Logistic Regression baseline variations

This notebook trains two variants of the same baseline model family:

1. `tfidf_logreg_basic`: TF-IDF over the raw context only.
2. `tfidf_logreg_marked`: TF-IDF over entity-marked context with section/source metadata.

The second version corresponds to the feature-engineering variation from the proposal.

In [10]:
def build_tfidf_logreg_pipeline() -> Pipeline:
    return Pipeline(
        steps=[
            (
                "tfidf",
                TfidfVectorizer(
                    lowercase=True,
                    ngram_range=(1, 3),
                    min_df=1,
                    max_df=0.95,
                    max_features=50_000,
                    sublinear_tf=True,
                    strip_accents="unicode",
                ),
            ),
            (
                "clf",
                LogisticRegression(
                    max_iter=2000,
                    C=10.0,
                    class_weight=None,
                    solver="lbfgs",
                    tol=1e-4,
                    random_state=RANDOM_SEED,
                ),
            ),
        ]
    )

def evaluate_predictions(y_true, y_pred, labels_order: list[str]) -> dict:
    labels_present = [label for label in labels_order if label in set(y_true) | set(y_pred)]
    return {
        "accuracy": accuracy_score(y_true, y_pred),
        "macro_f1": f1_score(y_true, y_pred, labels=labels_present, average="macro", zero_division=0),
        "weighted_f1": f1_score(y_true, y_pred, labels=labels_present, average="weighted", zero_division=0),
    }


def train_and_evaluate_variant(variant_name: str, text_column: str):
    print(f"\nTraining variant: {variant_name}")
    model = build_tfidf_logreg_pipeline()
    model.fit(train_df[text_column], train_df["label"])

    dev_pred = model.predict(dev_df[text_column])
    test_pred = model.predict(test_df[text_column])

    dev_metrics = evaluate_predictions(dev_df["label"], dev_pred, RELATIONSHIPS)
    test_metrics = evaluate_predictions(test_df["label"], test_pred, RELATIONSHIPS)

    metrics_row = {
        "variant": variant_name,
        "text_column": text_column,
        "dev_accuracy": dev_metrics["accuracy"],
        "dev_macro_f1": dev_metrics["macro_f1"],
        "dev_weighted_f1": dev_metrics["weighted_f1"],
        "test_accuracy": test_metrics["accuracy"],
        "test_macro_f1": test_metrics["macro_f1"],
        "test_weighted_f1": test_metrics["weighted_f1"],
        "train_examples": len(train_df),
        "dev_examples": len(dev_df),
        "test_examples": len(test_df),
    }

    # Save model.
    model_path = BASELINE_DIR / f"{variant_name}.joblib"
    joblib.dump(model, model_path)

    # Save classification report and confusion matrix for the test set.
    labels_present = [label for label in RELATIONSHIPS if label in set(test_df["label"]) | set(test_pred)]
    report = classification_report(
        test_df["label"],
        test_pred,
        labels=labels_present,
        output_dict=True,
        zero_division=0,
    )
    report_df = pd.DataFrame(report).transpose()
    report_df.to_csv(BASELINE_DIR / f"classification_report_{variant_name}.csv")

    cm = confusion_matrix(test_df["label"], test_pred, labels=labels_present)
    cm_df = pd.DataFrame(cm, index=[f"true_{x}" for x in labels_present], columns=[f"pred_{x}" for x in labels_present])
    cm_df.to_csv(BASELINE_DIR / f"confusion_matrix_{variant_name}.csv")

    # Save test predictions.
    test_out = test_df.copy()
    test_out["predicted_label"] = test_pred
    if hasattr(model.named_steps["clf"], "predict_proba"):
        probs = model.predict_proba(test_df[text_column])
        classes = model.named_steps["clf"].classes_
        test_out["confidence"] = probs.max(axis=1)
        for i, label in enumerate(classes):
            test_out[f"prob_{label}"] = probs[:, i]
    test_out.to_csv(BASELINE_DIR / f"predictions_test_{variant_name}.csv", index=False)

    return model, metrics_row


variants = {
    "tfidf_logreg_basic": "text_basic",
    "tfidf_logreg_marked": "text_marked",
}

trained_models = {}
metrics_rows = []
for variant_name, text_column in variants.items():
    model, metrics = train_and_evaluate_variant(variant_name, text_column)
    trained_models[variant_name] = {"model": model, "text_column": text_column}
    metrics_rows.append(metrics)

metrics_df = pd.DataFrame(metrics_rows).sort_values("dev_macro_f1", ascending=False).reset_index(drop=True)
metrics_df.to_csv(BASELINE_DIR / "metrics_baseline_variants.csv", index=False)

display(metrics_df)


Training variant: tfidf_logreg_basic

Training variant: tfidf_logreg_marked


,variant,text_column,dev_accuracy,dev_macro_f1,dev_weighted_f1,test_accuracy,test_macro_f1,test_weighted_f1,train_examples,dev_examples,test_examples
0,tfidf_logreg_basic,text_basic,0.936612,0.836208,0.933561,0.923699,0.818910,0.918881,4251,915,865
1,tfidf_logreg_marked,text_marked,0.933333,0.828982,0.930013,0.909827,0.800427,0.903640,4251,915,865


## 11. Choose the best baseline variant and predict all candidates

The best baseline variant is selected by dev macro-F1. Predictions for all candidate pairs are saved for later KG construction.

In [11]:
best_variant = metrics_df.iloc[0]["variant"]
best_text_column = metrics_df.iloc[0]["text_column"]
best_model = trained_models[best_variant]["model"]

print(f"Best variant by dev macro-F1: {best_variant} using {best_text_column}")

all_pred = candidates_df.copy()
all_pred["predicted_label"] = best_model.predict(all_pred[best_text_column])

if hasattr(best_model.named_steps["clf"], "predict_proba"):
    probs = best_model.predict_proba(all_pred[best_text_column])
    classes = best_model.named_steps["clf"].classes_
    all_pred["confidence"] = probs.max(axis=1)
    for i, label in enumerate(classes):
        all_pred[f"prob_{label}"] = probs[:, i]
else:
    all_pred["confidence"] = np.nan

structured_mask = (
    (all_pred["source_type"] == "infobox")
    & (all_pred["weak_label"] != NO_RELATION_LABEL)
)

all_pred.loc[structured_mask, "predicted_label"] = all_pred.loc[structured_mask, "weak_label"]
all_pred.loc[structured_mask, "confidence"] = 0.95

all_predictions_path = BASELINE_DIR / "predictions_all_best_baseline.csv"
all_pred.to_csv(all_predictions_path, index=False)
print(f"Saved all predictions to: {all_predictions_path}")
display(all_pred[["head", "tail", "label", "predicted_label", "confidence", "context"]].head(10))

Best variant by dev macro-F1: tfidf_logreg_basic using text_basic
Saved all predictions to: baseline\predictions_all_best_baseline.csv


,head,tail,label,predicted_label,confidence,context
0,Georgine,Sylvester,family,family,0.962552,"She is the mother of Detlinde, Sylvester's older sister and a former archduke candidate of Ehrenfest."
1,Magdalena,Werdekraf,romantic,family,0.746668,"She also often trained with her brother Werdekraf before she moved to live at the Royal Palace, following her marriage."
2,Arno,Ferdinand,enemy_rival,no_relation,0.744492,"Ferdinand, displeased at Arno's actions, had Arno killed."
3,Magdalena,Hildebrand,no_relation,no_relation,0.815832,The boy is Hildebrand and was raised to become a vassal to whichever of his two older brothers would win the position of successor.
4,Wilfried,Veronica,family,family,0.788611,Upon Veronica's imprisonment Wilfried was told that his grandmother had fallen ill and had been moved to a far away place to recover.
5,Lasfam,Ferdinand,service_retainer,service_retainer,0.488564,"When Ferdinand personally asked him to do as the others did so Lasfam wouldn't have to suffer so much, Lasfam refused, stating that doin..."
6,Eckhart,Lamprecht,family,family,0.837135,"Despite despising Veroncia as well, his father still didn't want this to come to pass and when he learned of Eckhart's plans intervened ..."
7,Lungtase,Raufereg,family,family,0.859618,"Her less well behaved brother Raufereg has to stay home, since his parents don't want to risk embarrassing themselves and by extension t..."
8,Gloria,Rozemyne,no_relation,no_relation,0.756899,"When Rozemyne enters noble society, Gloria holds her in contempt and often slanders her reputation at tea parties and social gatherings."
9,Georgine,Veronica,service_retainer,service_retainer,0.886851,"While Veronica was frozen in terror, one of her attendants had immediately sprung into action and administered an antidote."


## 12. Aggregate predictions into Knowledge Graph edges

This cell converts mention-level predictions into graph edges:

```text
head_character -- predicted_relationship --> tail_character
```

It removes `no_relation`, applies a confidence threshold, groups repeated evidence for the same pair/relation, and keeps the strongest relation for each pair.

In [12]:
# Relations such as family, romantic, friendship, and rivalry are treated as undirected:
# A --family--> B and B --family--> A are merged into one canonical edge.
# Service/retainer relations are treated as directed because direction matters:
# A --service_retainer--> B is not equivalent to B --service_retainer--> A.
UNDIRECTED_RELATIONS = {
    "family",
    "romantic",
    "friend_ally",
    "enemy_rival",
} & set(RELATIONSHIPS)

DIRECTED_RELATIONS = {
    "service_retainer",
} & set(RELATIONSHIPS)

RELATIONS_WITH_DIRECTION_RULES = UNDIRECTED_RELATIONS | DIRECTED_RELATIONS
UNSPECIFIED_RELATIONS = set(POSITIVE_RELATIONS) - RELATIONS_WITH_DIRECTION_RULES

if UNSPECIFIED_RELATIONS:
    print(
        "Warning: These positive relations are not listed in UNDIRECTED_RELATIONS or DIRECTED_RELATIONS "
        "and will be treated as directed:",
        sorted(UNSPECIFIED_RELATIONS),
    )

print(f"Undirected relations: {sorted(UNDIRECTED_RELATIONS)}")
print(f"Directed relations: {sorted(DIRECTED_RELATIONS)}")


def canonicalize_edge(row: pd.Series) -> pd.Series:
    """Create canonical edge keys.

    For undirected relations, sorted(head, tail) is used so mirrored predictions
    collapse into one edge. For directed relations, the original head/tail order is kept.
    """
    head = row["head"]
    tail = row["tail"]
    relation = row["predicted_label"]

    if relation in UNDIRECTED_RELATIONS:
        edge_head, edge_tail = sorted([head, tail])
        edge_direction = "undirected"
    else:
        # Directed relations and any unspecified positive relation keep model direction.
        edge_head, edge_tail = head, tail
        edge_direction = "directed"

    return pd.Series(
        {
            "edge_head": edge_head,
            "edge_tail": edge_tail,
            "edge_direction": edge_direction,
        }
    )


def aggregate_kg_edges(predictions: pd.DataFrame, confidence_threshold: float = EDGE_CONFIDENCE_THRESHOLD) -> pd.DataFrame:
    """Aggregate mention-level predictions into KG edges.

    Mirrored edges are merged for labels in UNDIRECTED_RELATIONS.
    Directed labels keep their original head -> tail direction.
    """
    edges = predictions.copy()
    edges = edges[edges["predicted_label"] != NO_RELATION_LABEL].copy()
    edges = edges[edges["confidence"].fillna(1.0) >= confidence_threshold].copy()

    empty_columns = [
        "head",
        "tail",
        "relation",
        "edge_direction",
        "evidence_count",
        "mean_confidence",
        "max_confidence",
        "edge_score",
        "evidence",
    ]

    if len(edges) == 0:
        return pd.DataFrame(columns=empty_columns)

    edge_keys = edges.apply(canonicalize_edge, axis=1)
    edges = pd.concat([edges, edge_keys], axis=1)

    grouped_rows = []
    group_cols = ["edge_head", "edge_tail", "predicted_label", "edge_direction"]

    for (edge_head, edge_tail, relation, edge_direction), group in edges.groupby(group_cols):
        evidence_count = len(group)
        mean_conf = float(group["confidence"].mean())
        max_conf = float(group["confidence"].max())
        edge_score = mean_conf * math.log1p(evidence_count)

        evidence = " | ".join(
            group
            .sort_values("confidence", ascending=False)["context"]
            .dropna()
            .astype(str)
            .drop_duplicates()
            .head(3)
            .tolist()
        )

        grouped_rows.append(
            {
                "head": edge_head,
                "tail": edge_tail,
                "relation": relation,
                "edge_direction": edge_direction,
                "evidence_count": evidence_count,
                "mean_confidence": mean_conf,
                "max_confidence": max_conf,
                "edge_score": edge_score,
                "evidence": evidence,
            }
        )

    kg_edges = pd.DataFrame(grouped_rows)

    # Keep only the highest-scoring relation per canonical character pair.
    # For undirected relations, this removes mirrored duplicates such as A-B and B-A.
    # For directed relations, the original direction is preserved.
    kg_edges = (
        kg_edges.sort_values(["head", "tail", "edge_score"], ascending=[True, True, False])
        .drop_duplicates(subset=["head", "tail"], keep="first")
        .sort_values("edge_score", ascending=False)
        .reset_index(drop=True)
    )

    return kg_edges


kg_edges_df = aggregate_kg_edges(all_pred)
kg_edges_path = BASELINE_DIR / "kg_edges_best_baseline.csv"
kg_edges_df.to_csv(kg_edges_path, index=False)

print(f"KG edges saved to: {kg_edges_path}")
print(f"Predicted nodes: {len(set(kg_edges_df['head']).union(set(kg_edges_df['tail'])) if len(kg_edges_df) else set())}")
print(f"Predicted edges: {len(kg_edges_df):,}")

if len(kg_edges_df):
    print("\nEdge direction counts:")
    display(kg_edges_df["edge_direction"].value_counts().rename_axis("edge_direction").reset_index(name="count"))

display(kg_edges_df.head(20))


Undirected relations: ['enemy_rival', 'family', 'friend_ally', 'romantic']
Directed relations: ['service_retainer']
KG edges saved to: baseline\kg_edges_best_baseline.csv
Predicted nodes: 201
Predicted edges: 720

Edge direction counts:


,edge_direction,count
0,undirected,610
1,directed,110


,head,tail,relation,edge_direction,evidence_count,mean_confidence,max_confidence,edge_score,evidence
0,Adolphine,Sigiswald,romantic,undirected,6,0.966772,0.984654,1.881252,"After graduating from the Royal Academy, she married Prince Sigiswald as his first wife. | Shortly before Adolphine and Prince Sigiswald..."
1,Gieselfried,Letizia,family,undirected,6,0.950000,0.950000,1.848615,"Infobox field family/Father: Drewanchel Archducal Family Member (Biological) Gieselfried (Adoptive, Deceased) | Infobox field family/Dau..."
2,Charlotte,Wilfried,family,undirected,5,0.972314,0.988042,1.742152,"Her older brother Wilfried was one year older than her, while her younger brother Melchior was four years younger. | Unlike her elder br..."
3,Elvira,Karstedt,romantic,undirected,5,0.969732,0.997272,1.737527,"Elvira’s husband Karstedt also married a second wife, Trudeliede, and a third wife, Rozemary. | He is married to Elvira, his First Wife,..."
4,Florencia,Sylvester,romantic,undirected,5,0.962538,0.985325,1.724637,"Florencia (フロレンツィア, Furorentsia) is the first wife of Sylvester, the archduke of Ehrenfest. | His second oldest sister Constanze married..."
5,Charlotte,Florencia,family,undirected,5,0.962114,0.987366,1.723876,"Unlike her elder brother Wilfried, she has been raised by her mother Florencia, and as a result, she is far better educated than her bro..."
6,Aeussewahl,Heileind,family,undirected,5,0.958338,0.971701,1.717111,He was the son of Aeussewahl and the brother of Tollkuehnheit. | Ultimately Aeussewahl chose his son Heileind as his successor. | Infobo...
7,Adelbert,Veronica,romantic,undirected,5,0.955188,0.969842,1.711468,"With his first wife Veronica, Adelbert had three children: Georgine, Constanze and Sylvester. | Adelbert went on to marry his cousin Ver..."
8,Adolphine,Ortwin,family,undirected,4,0.971259,0.996093,1.563181,"He is the younger brother of Adolphine. | Adolphine grew up closest to her younger brother Ortwin, since they shared the same mother. | ..."
9,Claudio,Elvira,family,undirected,4,0.967775,0.995825,1.557573,He is the older brother of Elvira. | He and his sister Elvira share a strong family resemblence. | Infobox field family/Sister: Elvira


## 13. Create PyVis Knowledge Graph visualization

If `pyvis` is installed, this cell creates an interactive HTML graph. If not, it writes a simpler HTML edge table so the pipeline still completes.

In [15]:
def inject_relation_filter(html_path: Path, relations: list[str]) -> None:
    """Inject relation checkbox filters into a PyVis HTML file."""
    html_path = Path(html_path)
    html_text = html_path.read_text(encoding="utf-8")

    checkbox_html = "\n".join(
        f"""
        <label style="display:block; margin: 3px 0;">
            <input type="checkbox" class="relation-filter" value="{relation}" checked>
            {relation}
        </label>
        """
        for relation in relations
    )

    control_panel = f"""
    <div id="relation-filter-panel" style="
        position: fixed;
        top: 10px;
        right: 10px;
        z-index: 9999;
        background: white;
        border: 1px solid #ccc;
        border-radius: 6px;
        padding: 10px 12px;
        font-family: Arial, sans-serif;
        font-size: 13px;
        box-shadow: 0 2px 8px rgba(0,0,0,0.15);
        max-width: 240px;
    ">
        <strong>Filter relations</strong>
        <div style="margin-top: 6px;">
            {checkbox_html}
        </div>
        <button id="select-all-relations" style="margin-top: 8px;">Select all</button>
        <button id="clear-all-relations" style="margin-top: 8px;">Clear all</button>
    </div>
    """

    filter_script = """
    <script type="text/javascript">
    document.addEventListener("DOMContentLoaded", function () {
        if (typeof edges === "undefined" || typeof nodes === "undefined" || typeof network === "undefined") {
            console.warn("PyVis variables not found; relation filter was not attached.");
            return;
        }

        var allEdges = edges.get();
        var allNodes = nodes.get();

        function getEdgeRelation(edge) {
            // Prefer custom relation metadata.
            // Fall back to edge label, because PyVis always keeps the label.
            return edge.relation || edge.label;
        }

        function updateGraphFilter() {
            var checkedRelations = Array.from(
                document.querySelectorAll(".relation-filter:checked")
            ).map(function (box) {
                return box.value;
            });

            var visibleNodeIds = new Set();
            var edgeUpdates = [];

            allEdges.forEach(function (edge) {
                var relation = getEdgeRelation(edge);
                var isVisible = checkedRelations.includes(relation);

                edgeUpdates.push({
                    id: edge.id,
                    hidden: !isVisible
                });

                if (isVisible) {
                    visibleNodeIds.add(edge.from);
                    visibleNodeIds.add(edge.to);
                }
            });

            var nodeUpdates = allNodes.map(function (node) {
                return {
                    id: node.id,
                    hidden: !visibleNodeIds.has(node.id)
                };
            });

            edges.update(edgeUpdates);
            nodes.update(nodeUpdates);

            network.redraw();
        }

        document.querySelectorAll(".relation-filter").forEach(function (box) {
            box.addEventListener("change", updateGraphFilter);
        });

        document.getElementById("select-all-relations").addEventListener("click", function () {
            document.querySelectorAll(".relation-filter").forEach(function (box) {
                box.checked = true;
            });
            updateGraphFilter();
        });

        document.getElementById("clear-all-relations").addEventListener("click", function () {
            document.querySelectorAll(".relation-filter").forEach(function (box) {
                box.checked = false;
            });
            updateGraphFilter();
        });
    });
    </script>
    """

    html_text = html_text.replace("<body>", f"<body>\n{control_panel}")
    html_text = html_text.replace("</body>", f"{filter_script}\n</body>")

    html_path.write_text(html_text, encoding="utf-8")

def write_pyvis_graph(edges_df: pd.DataFrame, output_path: Path, title: str = "TF-IDF + Logistic Regression KG"):
    """Write an interactive PyVis graph if pyvis is available; otherwise write a fallback HTML table."""
    output_path = Path(output_path)

    if len(edges_df) == 0:
        output_path.write_text("<html><body><h1>No edges passed the threshold.</h1></body></html>", encoding="utf-8")
        return "empty"

    try:
        from pyvis.network import Network

        net = Network(height="800px", width="100%", directed=True, notebook=False)
        net.barnes_hut()

        net.set_options("""
        {
          "physics": {
            "enabled": true,
            "barnesHut": {
              "gravitationalConstant": -25000,
              "centralGravity": 1,
              "springLength": 150,
              "springConstant": 0.04,
              "damping": 0.18,
              "avoidOverlap": 1
            },
            "stabilization": {
              "enabled": true,
              "iterations": 1200,
              "updateInterval": 25
            }
          },
          "nodes": {
            "font": {
              "size": 30
            }
          },
          "edges": {
            "font": {
              "size": 8
            },
            "smooth": {
              "enabled": true,
              "type": "dynamic"
            }
          },
          "interaction": {
            "dragNodes": true,
            "dragView": true,
            "zoomView": true
          }
        }
        """)

        degree_counter = Counter(edges_df["head"]) + Counter(edges_df["tail"])
        nodes = sorted(set(edges_df["head"]).union(set(edges_df["tail"])))

        for node in nodes:
            degree = degree_counter[node]
            net.add_node(
                node,
                label=node,
                size=min(35, 10 + 2 * degree),
                title=f"Character: {node}<br>Degree: {degree}",
            )

        for edge_idx, (_, row) in enumerate(edges_df.iterrows()):
            edge_direction = row.get("edge_direction", "directed")
            tooltip = (
                f"Relation: {row['relation']}<br>"
                f"Direction: {edge_direction}<br>"
                f"Mean confidence: {row['mean_confidence']:.3f}<br>"
                f"Evidence count: {int(row['evidence_count'])}<br>"
                f"Evidence: {row['evidence']}"
            )

            # Keep arrows only for directed relations such as service_retainer.
            # Undirected relations such as family, romantic, friend_ally, and enemy_rival
            # are displayed without arrowheads.
            if edge_direction == "undirected":
                arrows = {
                    "to": {"enabled": False},
                    "from": {"enabled": False},
                    "middle": {"enabled": False},
                }
            else:
                arrows = {
                    "to": {"enabled": True},
                    "from": {"enabled": False},
                    "middle": {"enabled": False},
                }

            net.add_edge(
                row["head"],
                row["tail"],
                label=row["relation"],
                width=0.5,
                arrows=arrows,
                title=tooltip,
                id=f"edge_{edge_idx}",
                relation=row["relation"],
                edge_direction=edge_direction,
            )

        net.write_html(str(output_path))
        relations = sorted(edges_df["relation"].dropna().unique().tolist())
        inject_relation_filter(output_path, relations)
        return "pyvis"

    except Exception as exc:
        fallback_html = f"""
        <html>
        <head><meta charset=\"utf-8\"><title>{title}</title></head>
        <body>
        <h1>{title}</h1>
        <p>PyVis graph could not be created: {html.escape(str(exc))}</p>
        {edges_df.to_html(index=False, escape=True)}
        </body>
        </html>
        """
        output_path.write_text(fallback_html, encoding="utf-8")
        return "fallback_html"


graph_path = BASELINE_DIR / "kg_tfidf_logreg_best_baseline.html"
graph_status = write_pyvis_graph(
    kg_edges_df,
    graph_path,
    title=f"Baseline KG: {best_variant}",
)

print(f"Graph status: {graph_status}")
print(f"Graph written to: {graph_path}")

Graph status: pyvis
Graph written to: baseline\kg_tfidf_logreg_best_baseline.html


## 14. Baseline output summary

This cell writes a compact JSON summary of the baseline run.

In [14]:
summary = {
    "xml_path": str(XML_PATH),
    "output_dir": str(BASELINE_DIR),
    "relationship_labels": RELATIONSHIPS,
    "num_pages": int(len(pages_df)),
    "num_candidates": int(len(candidates_df)),
    "num_train": int(len(train_df)),
    "num_dev": int(len(dev_df)),
    "num_test": int(len(test_df)),
    "split_method": split_method,
    "best_variant": str(best_variant),
    "best_text_column": str(best_text_column),
    "num_kg_edges": int(len(kg_edges_df)),
    "edge_confidence_threshold": float(EDGE_CONFIDENCE_THRESHOLD),
    "files_written": sorted(str(path) for path in BASELINE_DIR.glob("*")),
}

summary_path = BASELINE_DIR / "baseline_run_summary.json"
summary_path.write_text(json.dumps(summary, indent=2), encoding="utf-8")

print(json.dumps(summary, indent=2))

{
  "xml_path": "data\\bookworm_09062026.xml",
  "output_dir": "baseline",
  "relationship_labels": [
    "family",
    "romantic",
    "friend_ally",
    "service_retainer",
    "enemy_rival",
    "no_relation"
  ],
  "num_pages": 257,
  "num_candidates": 6031,
  "num_train": 4251,
  "num_dev": 915,
  "num_test": 865,
  "split_method": "grouped_by_pair",
  "best_variant": "tfidf_logreg_basic",
  "best_text_column": "text_basic",
  "num_kg_edges": 720,
  "edge_confidence_threshold": 0.95,
  "files_written": [
    "baseline\\baseline_run_summary.json",
    "baseline\\candidate_examples.csv",
    "baseline\\character_gazetteer.csv",
    "baseline\\classification_report_tfidf_logreg_basic.csv",
    "baseline\\classification_report_tfidf_logreg_marked.csv",
    "baseline\\confusion_matrix_tfidf_logreg_basic.csv",
    "baseline\\confusion_matrix_tfidf_logreg_marked.csv",
    "baseline\\dev.csv",
    "baseline\\kg_edges_best_baseline.csv",
    "baseline\\kg_tfidf_logreg_best_baseline.html",


## 15. What to report for this baseline

In your project report, use the files generated by this notebook to describe:

- the number of pages parsed from the XML,
- the number of candidate pairs,
- the label distribution,
- the difference between the basic TF-IDF baseline and the entity-marker/section-feature variation,
- macro-F1 and per-class F1 from `classification_report_*.csv`,
- the number of nodes and edges in `kg_edges_best_baseline.csv`, and
- qualitative graph observations from `kg_tfidf_logreg_best_baseline.html`.

Important limitation: this notebook uses weak labels. The scores are useful for checking the pipeline, but a final evaluation should use manually verified dev/test labels.